# Import per-well condition metadata

Merge per-well treatment / condition information (read from a user-supplied CSV) into the per-image analysis dataframe.

In [ ]:
df_path = Path(input())

In [ ]:
df = pd.read_csv(df_path)
df.head()

In [ ]:
img_basename = df['ROI imgname'].str.split('.ome.tif').str[0].str.replace('-', '_')
basename_splits = img_basename.str.split('_')
for i, elem in enumerate(basename_splits[0]):
    print(f'{i}: {elem}') 

In [ ]:
df['experiment'] = basename_splits.str[0]
df['scene'] = basename_splits.str[4]
df['wellID'] = df['experiment'] + '-' + 'I1' + '-' + basename_splits.str[3]
df['ROI'] = basename_splits.str[6]
df['UID'] = df['wellID'] + '_' + df['scene'] + '_' + df['ROI']
df.head()

df.at[0, 'UID']
#utils.safe_save_csv(df, df_path)

In [ ]:
wellcond_df_path = Path(input())

In [ ]:
wellcond_df = pd.read_csv(wellcond_df_path)
wellcond_df.head()


In [ ]:
len_df_premerge = len(df)
df = pd.merge(df, wellcond_df, how = "left", validate='many_to_one')
assert len_df_premerge == len(df)

df.head()

In [ ]:
# Note: first post-tx t counts frames from 1 instead of 0
df['tx status'] =  np.where((df['t'] < (df['first post-tx t']-1)), 'pre', 'post')
df.loc[:, 'elapsed time (hr)'] = (df['t'] - (df['first post-tx t']-1)) * df['time interval']

In [ ]:
front_cols = wellcond_df.columns.to_list()
front_cols.extend(['tx status', 'elapsed time (hr)'])
front_cols

In [ ]:
df = utils.move_columns_to_front(df, front_cols)
df.head()

In [ ]:
utils.safe_save_csv(df, df_path)